# TSIS &mdash; Lab 2
## Retrieval-Augmented Generation (RAG)

In the first lab, we gave a language model conversation history and tools. Today we will give it **relevant external knowledge**. We will build a small RAG system for a fictional software project, inspect what it retrieves, and then connect retrieval to an agent tool.

**Prerequisites:** make sure you've solved [the first lab](lab1_bis.ipynb) beforehand.


## Learning objectives

By the end of the lab you should be able to:

- explain the **indexing** and **query** stages of a RAG system;
- split **documents** into **chunks** and retrieve candidate matches based on **vector similarity**;
- distinguish **retrieval quality** (accuracy/precision) from **answer-generation quality** (the final output);
- store/query vectors and metadata in a [**vector database**](https://en.wikipedia.org/wiki/Vector_database) ([ChromaDB](https://www.trychroma.com/products/chromadb));
- ground an answer in retrieved **evidence** and make the model abstain when evidence is missing;
- expose **retrieval as a tool** that an agent can call.

RAG is an engineering pattern, with pros and cons. It can fail silently when the right evidence is not retrieved, when chunks lose context, or when the model ignores the evidence. We will explore some of these failure mods in this lab.

## What problem does RAG solve?

A model's weights are not a live database. They do not contain your private repository and they cannot learn a change made five minutes ago. RAG injects selected source material into the model's context at query time:

1. **Indexing (ahead of time):** load documents, split them into chunks, turn chunks into vectors, and store vectors plus metadata.
2. **Retrieval (per question):** turn the question into a vector and find nearby chunks.
3. **Generation:** put the question and retrieved chunks in a prompt and ask the model to answer from that evidence.

RAG trades infrastructure and a possible retrieval failure for a smaller, more focused context. Long-context prompting is a useful alternative when the complete source set is small or cross-document comparison matters. Production systems often combine both approaches, add metadata filters and reranking, and measure retrieval separately from generation.

## Setup and the local corpus

The notebook uses the same OpenAI-compatible interface as `lab1_bis.ipynb`. If you run into issues, ask the teaching assistant for help.

In [ ]:
%pip install --quiet --upgrade openai scikit-learn chromadb

In [ ]:
import json
from pathlib import Path

from pydantic import BaseModel

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "rag"

class Document(BaseModel):
    source: str
    text: str

documents: list[Document] = []
for path in sorted(DATA_DIR.glob("*.md")):
    document = Document(source=path.name, text=path.read_text(encoding="utf-8"))
    documents.append(document)

print(f"Loaded {len(documents)} documents:", [d.source for d in documents])
print(documents[0].text[:300])

## Chunking: deciding what can be retrieved

A whole document is often too broad to retrieve precisely, while tiny chunks lose context. There is no universally correct chunk size: headings, paragraphs, token counts, overlap, document structure, and the downstream question all matter.

For this small corpus, a paragraph is a useful and inspectable chunk. The metadata lets us show students exactly where an answer came from.

In [ ]:
class DocumentChunk(BaseModel):
    id: str
    source: str
    chunk_index: int
    text: str


def chunk_documents(documents: list[Document]) -> list[DocumentChunk]:
    chunks = []
    for document in documents:
        paragraphs = [p.strip() for p in document.text.split("\n\n") if p.strip()]

        for index, paragraph in enumerate(paragraphs):
            chunk = DocumentChunk(
                id=f"{document.source}#{index}",
                source=document.source,
                chunk_index=index,
                text=paragraph,
            )
            chunks.append(chunk)

    return chunks

chunks: list[DocumentChunk] = chunk_documents(documents)
for chunk in chunks:
    print(chunk.id, "->", chunk.text[:100].replace("\n", " "))

## A transparent local vector index

To keep the first implementation understandable, we use [TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) (_term frequency&ndash;inverse document frequency_) vectors. TF-IDF is not a modern semantic embedding model: it mainly matches words and phrases. That is a feature for teaching because we can inspect the representation and avoid requiring a second model endpoint. Cosine similarity gives us a ranked list of candidate chunks.

In a production system, an embedding model is normally chosen and versioned separately from the generation model. The rest of the retrieval interface can stay largely the same.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
matrix = vectorizer.fit_transform([chunk.text for chunk in chunks])

class SearchResult(DocumentChunk):
    score: float

def retrieve(query, k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, matrix)[0]
    ranked = scores.argsort()[::-1][:k]
    return [
        SearchResult(**chunks[index].model_dump(), score=float(scores[index]))
        for index in ranked
    ]

def show_results(results):
    for result in results:
        print(f"{result.score:.3f}  {result.id}\n{result.text}\n")

show_results(retrieve("What database is used in production?"))

## Mission 1 — Make retrieval reject weak evidence (guided)

**Goal:** prevent the system from presenting an unrelated chunk as evidence for an unknown question.

Complete the two blanks in the next cell:

1. choose a sensible default for `min_score` (maybe look at some examples);
2. return only results whose similarity score reaches that threshold.

Then run the checker. Try the known database question and the unknown cloud-provider question. Do not aim for a magic threshold: inspect the scores and explain your choice.


In [ ]:
def retrieve_confidently(query, k=3, min_score=___):
    candidates = retrieve(query, k=k)
    raise NotImplementedError("Fill in the blank with a minimum score threshold for confident retrieval.")
    return [candidate for candidate in candidates if candidate.score >= ___]


def check_confident_retrieval():
    known = retrieve_confidently("Which database is supported in production?")
    unknown = retrieve_confidently("Which cloud provider hosts the production cluster?")
    assert known, "The threshold rejected the known answer. Lower it or inspect the scores."
    assert len(unknown) < len(retrieve("Which cloud provider hosts the production cluster?")), (
        "The threshold did not reject any weak evidence. Inspect the scores."
    )
    print("Known question:", [(r.id, round(r.score, 3)) for r in known])
    print("Unknown question:", [(r.id, round(r.score, 3)) for r in unknown])

# Complete the blanks, then uncomment the next line.
check_confident_retrieval()

**Discuss:** should a production system return no evidence, ask a clarification question, or fall back to a broader search? What additional signal would you use besides a similarity threshold?

**Observe** how retrieval returns candidates (raw document chunks), not formatted answers. Try a synonym or an unanswerable question and inspect the scores. A top result with a low score is not evidence that the corpus contains the answer.

**Exercise 1 (5 minutes):** change the chunking function so that adjacent short paragraphs are combined. Does the top result improve for a question that needs two facts? What context is lost when chunks become too large?

## The same index in a vector database

A vector database stores vectors together with IDs, documents, and metadata, then handles persistence, filtering, and nearest-neighbor queries. Here Chroma receives the vectors we already computed, so it does not download or choose an embedding model behind our backs. Real deployments commonly use a hosted embedding service or a local embedding model and configure an approximate-nearest-neighbor index.


In [ ]:
import chromadb

chroma = chromadb.Client()
collection = chroma.get_or_create_collection(
    name="atlas-handbook",
    configuration={"hnsw": {"space": "cosine"}},
)
collection.add(
    ids=[chunk.id for chunk in chunks],
    documents=[chunk.text for chunk in chunks],
    metadatas=[{"source": chunk.source, "chunk_index": chunk.chunk_index} for chunk in chunks],
    embeddings=matrix.toarray().tolist(),
)

def retrieve_from_database(query, k=3):
    query_vector = vectorizer.transform([query]).toarray().tolist()
    result = collection.query(query_embeddings=query_vector, n_results=k)
    return [
        SearchResult(
            id=result["ids"][0][i],
            text=result["documents"][0][i],
            source=result["metadatas"][0][i]["source"],
            score=1 - result["distances"][0][i],
            chunk_index=result["metadatas"][0][i]["chunk_index"]
        )
        for i in range(len(result["ids"][0]))
    ]

show_results(retrieve_from_database("How are users authenticated?"))

## Mission 2 — Improve the chunks (guided)

**Goal:** change the indexing strategy and measure whether it helps retrieval.

The baseline treats every paragraph as a chunk. Implement `chunk_documents_by_words` below:

- preserve the `DocumentChunk` shape and source metadata;
- create windows of `chunk_size` words;
- overlap adjacent windows by `overlap` words; and
- make sure the loop always advances (try `chunk_size=45, overlap=10`).

After implementing it, build a second TF-IDF index and compare the retrieval report from Section 7. Which questions improved, and which became worse?


In [ ]:
def chunk_documents_by_words(
    documents: list[Document], chunk_size: int = ___, overlap: int = ___
) -> list[DocumentChunk]:
    """Create overlapping word windows while preserving source metadata."""
    if not 0 < overlap < chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    result = []
    for document in documents:
        words = document.text.split()
        start = 0
        chunk_index = 0
        while start < len(words):
            raise NotImplementedError("Fill in the blanks to create overlapping word windows.")
            # TODO: choose the end of this window and create a DocumentChunk.
            end = ___
            result.append(DocumentChunk(
                id=f"{document.source}#window-{chunk_index}",
                source=document.source,
                chunk_index=chunk_index,
                text=" ".join(words[start:end]),
            ))
            chunk_index += 1
            # TODO: move to the next overlapping window.
            start = ___
    return result

# Implement the blanks, then run:
windowed_chunks = chunk_documents_by_words(documents)
print(len(chunks), "paragraph chunks ->", len(windowed_chunks), "word-window chunks")

**Extension:** add the Markdown heading to each chunk, or keep a one-sentence overlap instead of a word overlap. Explain why your strategy might help a real codebase or requirements repository.

A production vector store is more than a Python dictionary: it usually offers durable storage, collection/index management, metadata filters, concurrent updates, and operational metrics. Examples include Chroma, pgvector, Qdrant, Milvus, and managed search services. The important design boundary is still the same: **source documents → chunks → embeddings → searchable store**.

Notice that vector similarity is not the same as truth. A system may retrieve a plausible but irrelevant chunk. A score threshold, a reranker, a keyword/metadata filter, or a request for more context can help—but each adds a design and evaluation cost.

## Grounded generation through vLLM

The prompt should tell the model what counts as evidence and what to do when the evidence is insufficient. We also include source labels so that the answer can be inspected.

In [ ]:
from openai import OpenAI

BASE_URL = "http://localhost:27182/v1"
MODEL = "Qwen/Qwen3.8-27B-FP8"
client = OpenAI(api_key="EMPTY", base_url=BASE_URL)

def build_context(results):
    return "\n\n".join(
        f"[Source: {result.source}, chunk {result.chunk_index}]\n{result.text}"
        for result in results
    )

def answer_with_rag(question, k=3):
    results = retrieve_from_database(question, k=k)
    prompt = f"""You answer questions about the Atlas project.
Use only the evidence between <evidence> tags. If the evidence does not answer the question, say exactly that the handbook does not provide enough information. Do not invent details. Cite the source filename in square brackets.

<evidence>
{build_context(results)}
</evidence>

Question: {question}"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content, results

question = "How quickly must an on-call engineer acknowledge an alert?"

answer, evidence = answer_with_rag(question)
print("Question:", question)
print(answer)
print("\nRetrieved evidence:")
show_results(evidence)

**Exercise 2 (10 minutes):** ask the system about the cloud provider. Compare the answer with the retrieved chunks. Then remove the abstention instruction and ask again. What changed? This is a small example of why prompt grounding and an explicit unknown policy matter, even though neither can compensate for missing retrieval evidence.

## Evaluation and failure analysis

We can evaluate retrieval without judging prose: did the expected source appear in the top-k results? This is a tiny labelled fixture, not a statistically meaningful benchmark, but it gives us a repeatable regression check. In a real system, expand the question set, define relevance carefully, and track recall@k, precision@k, latency, cost, and answer faithfulness.

In [ ]:
class Question(BaseModel):
    question: str
    expected_source: str | None
    answerable: bool

questions = json.loads((DATA_DIR / "questions.json").read_text(encoding="utf-8"))
questions = [Question(**item) for item in questions]

def retrieval_report(k=3):
    rows = []
    for item in questions:
        results = retrieve_from_database(item.question, k=k)
        sources = {result.source for result in results}
        hit = item.expected_source in sources if item.expected_source else True
        rows.append({"question": item.question, "expected": item.expected_source, "hit": hit})
    return rows

report = retrieval_report(k=3)
for row in report:
    print(f"{'PASS' if row['hit'] else 'MISS'} | expected={row['expected']} | {row['question']}")
print(f"Recall-like score: {sum(row['hit'] for row in report)}/{len(report)}")

# Inspect the unanswerable question instead of treating a top-1 result as proof.
show_results(retrieve_from_database("Which cloud provider hosts the production cluster?", k=3))

## Mission 3 — Make the RAG prompt testable (guided)

**Goal:** make the model's grounding contract explicit and inspectable.

Fill in the prompt template below. It must:

- identify the supplied text as the only evidence;
- tell the model what to say when the evidence is insufficient; and
- request a citation using the source filename.

The checker is intentionally simple: it verifies that the required instructions and the question/evidence are present. Then use your prompt in a second generation call and compare it with `answer_with_rag`.


In [ ]:
RAG_PROMPT_TEMPLATE = """You answer questions about the Atlas project.

Evidence:
{evidence}

Question: {question}

Answer: ___"""


def build_testable_prompt(question, results):
    prompt = RAG_PROMPT_TEMPLATE.format(
        evidence=build_context(results),
        question=question,
    )
    raise NotImplementedError("Fill in the blanks to create a testable prompt with evidence-only, insufficient-evidence, and source-citation instructions.")
    required_phrases = [___, ___, ___]
    assert all(phrase.lower() in prompt.lower() for phrase in required_phrases), (
        "Include an evidence-only rule, an insufficient-evidence instruction, "
        "and a source-citation instruction."
    )
    return prompt

# Complete the template and required_phrases, then try:
print(build_testable_prompt("Which cloud provider hosts the production cluster?", retrieve_from_database("cloud provider")))

**Extension:** add a rule that distinguishes a source citation from a claim that merely sounds plausible. What would you test automatically in a production answer evaluator?

Typical failure modes to investigate:

- **chunking failure:** the answer is split across chunks or its heading is separated from its body;
- **representation failure:** TF-IDF misses a synonym that a semantic embedding might match;
- **ranking failure:** a related but wrong chunk appears above the useful chunk;
- **coverage failure:** the source simply does not contain the answer; and
- **generation failure:** the model ignores, overgeneralizes, or contradicts supplied evidence.

**Exercise 3 (10 minutes):** change `k` between 1 and 5, and compare the retrieval report. Which question is most sensitive to `k`? In a production design, would you change chunking, embeddings, reranking, metadata filters, or the prompt first? Explain your choice.

## RAG as an agent tool

Instead of putting retrieval in every prompt-building function, an agent can decide when it needs the handbook. This is the same control boundary as Lab 1 tool calling: the model proposes a tool call, but application code executes retrieval and returns the result.

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_atlas_handbook",
        "description": "Search the Atlas project handbook for evidence relevant to a question.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}]

def ask_agent(question):
    messages = [{"role": "user", "content": question}]
    first = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
    message = first.choices[0].message
    if not message.tool_calls:
        return message.content

    messages.append(message)
    for call in message.tool_calls:
        if call.function.name != "search_atlas_handbook":
            raise ValueError(f"Unexpected tool: {call.function.name}")
        arguments = json.loads(call.function.arguments)
        results = retrieve_from_database(arguments["query"], k=3)
        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": build_context(results),
        })
    final = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    return final.choices[0].message.content

question = "What is the rollback policy when the error rate is high?"
print("Question:", question)

# The model may answer directly or call the retrieval tool depending on its tool-choice decision.
print(ask_agent(question))

## Takeaways

- RAG is a pipeline: ingest, chunk, represent, retrieve, augment, and generate.
- Retrieval is an independent component that needs its own tests and metrics.
- Vector databases store vectors plus metadata; they do not make an embedding model or a prompt automatically correct.
- Grounded prompts should define evidence boundaries and an unknown/abstention behavior.
- Long-context prompting can be simpler for small, complete corpora, while RAG reduces repeated context processing and focuses attention on relevant material.
- In an agentic system, retrieval can be a controlled tool: the application owns the data access and the model decides when to request it.

For further work, replace TF-IDF with a versioned embedding model, add metadata filters and reranking, persist Chroma to a directory, and build a larger labelled evaluation set.